# OTOCs of the memory under the ATS pump (poster parameters)

The out-of-time-order correlator `C(t) = -<[x_a(t), p_a(0)]^2>` of a
two-photon-pumped ATS (memory `a`, buffer `b` with single-photon loss
`kappa_b`), in the cat `|i alpha> + |-i alpha>` (buffer in vacuum). `x_a(t)` is
the Heisenberg-picture position, in the rotating frame of the memory, under the
full Lindbladian: the solution of the adjoint master equation

`dX/dt = i[H, X] + kappa_b (b^dag X b - 1/2 {b^dag b, X})`.

The growth rate is a linear fit of `log C` against `kappa_b t`.

The circuit is the poster one (`w_a = 25.34 rad/ns`, `w_b = 2 w_a`, `phi_b =
0.204`, `E_J = 1.5 x 37.12 x 2pi`, `kappa = 2pi / 10.4`).

How `x_a(t)` is computed (details in `otoc_point`):

- The pump is static (`w_p = 2 w_a - w_d = 0`), so the Lindbladian is periodic
  with the drive period `tau = 2 pi / w_b`, and even in time. Integrating the
  adjoint equation forward reverses the time ordering, which is exact only at
  `t = n tau`. With the drive phase shifted by `r`, one solve gives `x_a(r + n
  tau)` exactly for every `n`. The time grid is `n_sub` such solves, one per
  offset `r = k tau / n_sub`.
- The solves run in the interaction frame of the static Hamiltonian, as in
  Floquet-Lindblad's `_interaction_frame_ops`.
- The adjoint equation is integrated by the dynamiqs fork's `mesolve_fast`
  (precomputed `L^dag L`) with `H -> -H`, `L -> L^dag` and `L^dag L` passed
  explicitly. No change to dynamiqs is needed, but the fork is shipped with the
  job.

Until 2026-10-12 this notebook used `dq.mesolve(-H, [identity], x_a, tsave)`.
That had no loss at all (an identity jump operator contributes nothing), the
wrong time ordering between drive periods, and the lab-frame `x_a`. Its
results (`m = 28.8` chaotic, `2.42` stabilized) are not comparable with the
current ones.

This notebook used to run on `chuc`, on four local GPUs (`jax.pmap` for the
heatmap), with a `Plot_setting` module from a home directory. It now runs on the
anb-compute Ray cluster. Each simulation (`otoc_point`) is one GPU task, which
returns the OTOC curve `C(t)` as plain numpy. The fits and the plots happen
here. The original is kept in `legacy_chuc/`.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`).
   These must match the cluster.
2. **Vault session**, once (lasts about a week):

   ```bash
   export VAULT_ADDR=https://vault.int.alice-bob.com
   vault login -method=oidc role=research_core_user
   ```

3. Run all cells. The first `acr.run` of a session may open an interactive
   Microsoft device-code login for GCS read access. That is expected, and the
   login is cached.

If you have already run something that imports `ray` in this kernel, restart it
first. Ray caches `RAY_AUTH_MODE` at first import.

The GPU image's `prod-gpu` environment already has jax (CUDA) and scipy. Its
dynamiqs 0.3.6 has no `mesolve_fast`, so each job ships the fork (upstream
0.3.6 + `mesolve_fast`, the one Floquet-Lindblad pins) from the checkout at
`/workspace/dynamiqs`, branch `precomputed-ldl-v036` (override with
`$DYNAMIQS_SRC`). Uncommitted changes in that checkout ship too. The functions
below travel by value.


In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import ray

from anb_compute import ray as acr

print("python", sys.version.split()[0], "| ray", ray.__version__)

In [ ]:
import numpy as np

# --- constants (computed once) ---
ħ = 1
e = 1
phi_quantum = 1/2

w_a = 25.338776456203686 #GHz
w_b = 2*w_a


phi_a = 0.11
Z_a = 2 * phi_quantum**2 * phi_a**2
C_a_inv = Z_a * w_a
E_Ca = C_a_inv / 2

phi_b = 0.204
Z_b = 2 * phi_quantum**2 * phi_b**2
C_b_inv = Z_b * w_b
E_Cb = C_b_inv / 2

E_J = 37.12 * 2 * np.pi * 1.5
kappa = 2*np.pi/10.4

# dynamiqs' default, i.e. what the original ran with. "double" for complex128
# throughout.
PRECISION = "single"


### One OTOC curve, on one GPU worker

`otoc_point` returns `C(t)` on `t = j tau / n_sub`, `j = 0 .. n_periods n_sub`,
for one point. It runs `n_sub` adjoint solves, so it costs about `n_sub` solves
over `n_periods` drive periods. Only `C(t)` comes back: the Heisenberg operators
stay on the worker. jax and dynamiqs are imported inside, on the worker.

It evolves `a` rather than `x_a`, which gives both the rotating-frame position
`(a e^{i w_a t} + h.c.) / sqrt(2)` (`"otoc"`, the one fitted) and the lab-frame
one (`"otoc_lab"`). The lab-frame `x_a` turns at `w_a`, so its `C(t)` oscillates
within a drive period. The two agree at `t = n tau`.

The two sections differ only in their parameters: `epsilon_d = ratio g2`, the
cat amplitude `alpha = sqrt(epsilon_d / (2 g2)) = sqrt(ratio / 2)` (`sqrt(5)` at
the regime sections' `ratio = 10`), and `g2 = sin(epsilon_0) E_J phi_a^2 phi_b`.
`g2` is computed on the worker or, for the heatmap, passed in from a float32
`epsilon_0`, as in the original. The cat is now normalized (`dq.unit`). The
original's norm was off by `1 + exp(-2 alpha^2)`, 13 % at `ratio = 2`.

`g2` has `sin(epsilon_0)` because the pump is static. `J_1(epsilon_0)` in its
place holds only for `w_p != 0`. It was used here until 2026-10-12, and
underestimated `g2` by about 2x (`J_1 / sin = 0.50` at `epsilon_0 = 0.2`,
`0.53` at `1.1`).

In [ ]:
def otoc_point(*, label, epsilon_0, ratio, n_a, n_b, n_periods, n_sub, kappa_b,
               g2=None, precision="single"):
    """C(t) = -<[x_a(t), p_a(0)]^2> for one point, on one GPU worker.

    x_a(t) is the Heisenberg-picture operator under the full Lindbladian (buffer
    loss kappa_b), on t = j tau / n_sub, j = 0 .. n_periods n_sub, with
    tau = 2 pi / w_b the drive period.
    """
    import time

    import jax
    import jax.numpy as jnp
    import numpy as np

    import dynamiqs as dq

    if not hasattr(dq, "mesolve_fast"):
        raise RuntimeError(f"dynamiqs at {dq.__file__} is not the fork with mesolve_fast; "
                           "ship it with py_modules (see run_on_cluster)")

    t_start = time.time()
    dq.set_precision(precision)
    dims = (n_a, n_b)

    if g2 is None:
        g2 = np.sin(epsilon_0) * E_J * phi_a**2 * phi_b  # static pump: sin, not J_1
    epsilon_d = ratio * g2

    a = dq.tensor(dq.destroy(n_a), dq.eye(n_b))
    b = dq.tensor(dq.eye(n_a), dq.destroy(n_b))
    varphi_a = jnp.sqrt(Z_a/2) * (a + a.dag()) / phi_quantum
    varphi_b = jnp.sqrt(Z_b/2) * (b + b.dag()) / phi_quantum

    # The pump is static (w_p = 2 w_a - w_d = 0), so the only time dependence is
    # the drive epsilon_d cos(w_b t) (b + b^dag). As in Floquet-Lindblad's
    # _interaction_frame_ops, everything is written in the eigenbasis of the
    # static Hamiltonian H_s = V diag(lam) V^dag.
    H_s = (w_a * a.dag() @ a + w_b * b.dag() @ b
           - 2 * E_J * jnp.sin(epsilon_0) * (dq.sinm(varphi_a + varphi_b) - varphi_a - varphi_b))
    Hs_mat = H_s.to_numpy().astype(np.complex128)
    lam, V = np.linalg.eigh(0.5 * (Hs_mat + Hs_mat.conj().T))
    Vd = V.conj().T

    def to_eig(op):
        return jnp.asarray(Vd @ op.to_numpy() @ V)

    a_bar, b_bar = to_eig(a), to_eig(b)
    bd_plus_b_bar = b_bar + b_bar.conj().T
    bdb_bar = b_bar.conj().T @ b_bar
    p_bar = to_eig(1j * (a.dag() - a) / jnp.sqrt(2))
    Delta = jnp.asarray(lam[:, None] - lam[None, :])

    alpha = jnp.sqrt(ratio / 2)
    cat = dq.unit(dq.coherent(n_a, 1j*alpha) + dq.coherent(n_a, -1j*alpha))
    psi_bar = jnp.asarray(Vd @ dq.tensor(cat, dq.fock(n_b, 0)).to_numpy()[:, 0])

    # Heisenberg picture: A_H(t) = Phi_t^dag(A), Phi_t the Lindblad channel from 0
    # to t. Integrating the adjoint equation dY/du = L^dag(s - u) Y from u = 0
    # gives Phi_s^dag at u = s (time ordering reversed). L(t) has period tau and
    # is even, so with the drive phase shifted by r, one solve gives A_H(r + n tau)
    # for every n. Frame: Y = W Y~ W^dag, W = exp(i H_s u), which turns every
    # operator O into O_bar * exp(-1j Delta u).
    #
    # mesolve_fast applies -i[H', Y] + L' Y L'^dag - 1/2 {LdL', Y}. The adjoint
    # equation i[H, Y] + L^dag Y L - 1/2 {L^dag L, Y} is H' = -H, L' = L^dag and
    # LdL' = L^dag L, passed explicitly (L' L'^dag would be L L^dag).
    def frame(u):
        return jnp.exp(-1j * Delta * u)

    def make_H_adj(r):
        return dq.timecallable(lambda u: dq.asqarray(
            -epsilon_d * jnp.cos(w_b * (r - u)) * bd_plus_b_bar * frame(u), dims=dims))

    L_adj = dq.timecallable(lambda u: dq.asqarray(
        jnp.sqrt(kappa_b) * (b_bar * frame(u)).conj().T, dims=dims))
    LdL_adj = dq.timecallable(lambda u: dq.asqarray(kappa_b * bdb_bar * frame(u), dims=dims))

    tau = 2 * np.pi / w_b
    t_list, otoc_list, otoc_lab_list = [], [], []
    for k in range(n_sub):
        r = k * tau / n_sub
        us = r + tau * np.arange(n_periods + 1 if k == 0 else n_periods)
        res = dq.mesolve_fast(make_H_adj(r), [L_adj], [LdL_adj], dq.asqarray(a_bar, dims=dims),
                              us, t0=0.0, assume_hermitian=False, progress_meter=False)
        for u, a_tilde in zip(us, res.states.to_jax()):
            a_t = a_tilde * jnp.exp(1j * Delta * u)  # a_H(t), eigenbasis of H_s
            a_rot = a_t * jnp.exp(1j * w_a * u)      # rotating frame of the memory
            for x_list, a_op in ((otoc_list, a_rot), (otoc_lab_list, a_t)):
                x = (a_op + a_op.conj().T) / jnp.sqrt(2)
                # [x, p] is anti-Hermitian, so -<[x, p]^2> = ||[x, p] psi||^2
                comm_psi = x @ (p_bar @ psi_bar) - p_bar @ (x @ psi_bar)
                x_list.append(float(jnp.vdot(comm_psi, comm_psi).real))
        t_list.extend(us)

    order = np.argsort(t_list)
    return {
        "label": label,
        "epsilon_0": float(epsilon_0),
        "ratio": float(ratio),
        "g2": float(g2),
        "epsilon_d": float(epsilon_d),
        "kappa_b": float(kappa_b),
        "n_a": n_a,
        "n_b": n_b,
        "tsave": np.asarray(t_list)[order],
        "otoc": np.asarray(otoc_list)[order],          # rotating-frame x_a
        "otoc_lab": np.asarray(otoc_lab_list)[order],  # lab-frame x_a
        "device": str(jax.devices()[0]),
        "elapsed_s": time.time() - t_start,
    }

### The coordinator

One job per section, one GPU task per point. At most `N_MAX_GPUS` tasks, so at
most `N_MAX_GPUS` GPUs, are in flight at once: the coordinator submits the first
`N_MAX_GPUS` points, then the next one each time a task finishes. Pass
`n_max=...` to `run_on_cluster` to override it for one call. The coordinator
only fans out and gathers, so it is submitted with `entrypoint_num_cpus=0`, which keeps it on the
head node rather than on a freshly started spot worker (see
`/workspace/ray_driver_placement.md`). If that ever misbehaves, drop the
argument: the job then runs exactly as `acr.run`'s default.

The job ships the dynamiqs fork (`py_modules`), and the tasks it fans out
inherit it.

Each point is caught on its own: a task lost to the infrastructure (node
preempted, OOM) comes back as an `"error"` entry instead of sinking the rest.

In [ ]:
import pathlib
import time

# Per GPU task. One whole GPU each: jax preallocates most of the card's memory.
TASK_RESOURCES = dict(num_gpus=1, num_cpus=2)
# At most this many tasks in flight at once, i.e. at most this many GPUs taken on
# the cluster. The other points wait in the coordinator, not in Ray's queue.
N_MAX_GPUS = 32
# The dynamiqs fork with mesolve_fast, the one Floquet-Lindblad pins. The image's
# dynamiqs 0.3.6 does not have it, so every job ships it.
_here = pathlib.Path.cwd()
DYNAMIQS_SRC = os.environ.get("DYNAMIQS_SRC") or str(
    (_here.parent if _here.name == "Quantum_chaos" else _here) / "dynamiqs" / "dynamiqs")


def sweep(worker, points, common, resources, n_max):
    """Coordinator: one GPU task per point, at most `n_max` running at a time.

    `worker` is passed in because it is defined in this notebook and ships by
    value. Results come back in the order of `points`.
    """
    import ray

    from anb_compute import ray as acr

    n_max = max(1, int(n_max))
    out = [None] * len(points)
    next_i = 0
    refs = {}  # in-flight task -> its index in points
    while next_i < len(points) or refs:
        while next_i < len(points) and len(refs) < n_max:
            refs[acr.submit(worker, **resources, **common, **points[next_i])] = next_i
            next_i += 1
        ready, _ = ray.wait(list(refs), num_returns=1)
        i = refs.pop(ready[0])
        try:
            res = ray.get(ready[0])
        except Exception as exc:  # a task lost to infra, not to numerics
            res = {**points[i], "error": f"{type(exc).__name__}: {exc}"}
        out[i] = res
        print(f"{res['label']:<28} "
              + (res["error"] if "error" in res else f"{res['elapsed_s']:.1f} s"))
    return out


def run_on_cluster(worker, points, common, n_max=N_MAX_GPUS):
    t0 = time.time()
    results = acr.run(
        sweep, worker, points, common, TASK_RESOURCES, n_max,
        runtime_env=acr.build_runtime_env(py_modules=[DYNAMIQS_SRC]),
        poll_seconds=5,
        entrypoint_num_cpus=0,  # coordinator on the head, never on a spot node
    )
    print(f"\nwall clock, incl. job startup: {time.time() - t0:.1f} s "
          f"for {len(results)} point(s)")
    for r in results:
        if "error" in r:
            print(f"FAILED {r['label']}: {r['error']}")
    return results

In [ ]:
import matplotlib.pyplot as plt


def fit_log_otoc(res):
    """Linear fit of log C(t) against kappa_b t: slope m and R^2, as in the original."""
    t_kappa = res["tsave"] * kappa
    log_otoc = np.log(res["otoc"])
    (m, _), residuals, _, _, _ = np.polyfit(t_kappa, log_otoc, 1, full=True)
    SS_tot = sum((log_otoc - np.mean(log_otoc))**2)
    R_sq = 1 - residuals / SS_tot
    return m, R_sq.item()

# Chaotic and stabilized regimes

`epsilon_0 = 0.93` (chaotic) and `epsilon_0 = 0.2` (stabilized), at
`epsilon_d = 10 g2`, `n_a = 38`, buffer loss `kappa_b = kappa`, in one job. The
time grid is 8 points per drive period over 6 periods, `kappa_b T = 0.225` (the
original's `T = 0.2 / kappa`, rounded up to whole periods). Before the switch to
the adjoint equation, the closed-system runs here gave `m = 28.8` (`R^2 = 0.790`)
and `m = 2.42` (`R^2 = 0.0174`). Both curves are saved, as in the original
(`otoc_example_<date>.json`, `otoc_example_stable_<date>.json`).

In [ ]:
# time grid: n_sub points per drive period tau = 2 pi / w_b, over n_periods periods
tau = 2 * np.pi / w_b
n_periods = int(np.ceil(.2 / kappa / tau))   # 6: kappa_b T = 0.225
n_sub = 8   # one adjoint solve per sub-period offset
n_a = 38  # Example value for n_a
n_b = (n_a + 1) // 2

REGIMES = [
    dict(label="chaotic, eps_0 = 0.93", epsilon_0=0.93),
    dict(label="stabilized, eps_0 = 0.2", epsilon_0=0.2),
]
COMMON = dict(ratio=10.0, n_a=n_a, n_b=n_b, n_periods=n_periods, n_sub=n_sub,
              kappa_b=kappa, precision=PRECISION)

regimes = run_on_cluster(otoc_point, REGIMES, COMMON)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True)
for ax, res in zip(axes, regimes):
    if "error" in res:
        ax.set_title(f"{res['label']}: FAILED", fontsize=9)
        continue
    m, R_sq = fit_log_otoc(res)
    ax.plot(res["tsave"] * kappa, res["otoc"], label="rotating frame (fitted)")
    ax.plot(res["tsave"] * kappa, res["otoc_lab"], alpha=0.4, label="lab frame")
    ax.legend(fontsize=8)
    ax.set_yscale('log')
    ax.set_ylabel(r'$\langle[x_a(t),p_a(0)]^2\rangle$')
    ax.set_xlabel(r'$\kappa_b t$')
    ax.set_title(f"OTOC, {res['label']}: " + r'$n_a=$' + str(res['n_a']), fontsize=9)
    print(f"{res['label']:<28} m = {m:.6g}   R^2 = {R_sq:.6g}")
plt.show()

In [ ]:
import datetime
import json
import pathlib

# Written next to this notebook (or the workspace root, if that is the kernel's
# working directory) under results/.
here = pathlib.Path.cwd()
out_dir = (here if here.name == "Quantum_chaos" else here / "Quantum_chaos") / "results"
out_dir.mkdir(parents=True, exist_ok=True)
date = f"{datetime.date.today():%d_%m_%Y}"

# Same format as the original: the list of C(t) values on tsave.
for name, res in (("otoc_example", regimes[0]), ("otoc_example_stable", regimes[1])):
    if "error" in res:
        print(f"{res['label']} failed, not saved")
        continue
    out_path = out_dir / f"{name}_{date}.json"
    with open(out_path, "w") as f:
        json.dump(res["otoc"].tolist(), f)
    print(f"wrote {out_path}")

# Heatmap

The fitted rate over `epsilon_0` and `epsilon_d / g2 = 2, ..., 9`, at
`n_a = 29`, `n_b = 17`, buffer loss `kappa_b = kappa`, on the same time grid
as above: one GPU task per point, 184 in all,
at most `N_MAX_GPUS` of them running at once. (The original
batched all of them with `pmap` x `vmap` and ran out of GPU memory, asking for
9.4 GB per card. One point per task avoids that.) As in the original,
`epsilon_0` is cast to float32 before `g2` is computed, and the plotted rate is
`lyap = m / 2`. `end_otoc_arr` is `C(T)`, the last point of each curve.

`lyap_arr[i, j]`, `R_sq_arr[i, j]` and `end_otoc_arr[i, j]` are at `epsilon_0_list[i]`,
`ratio_list[j]`, the same layout as the original's arrays. A failed point is
`nan`. The fit now runs here in float64 numpy rather than in float32 jax on the
GPU, so the last digits can differ from the original.

In [ ]:
# Precompute with NumPy/Scipy
epsilon_0_list = list(np.concatenate((np.arange(0.2, .6, 0.07), np.arange(0.6, 1.1, 0.03))))
epsilon_0_arr = np.array(epsilon_0_list, dtype=np.float32)
g2_arr = np.sin(epsilon_0_arr) * E_J * phi_a**2 * phi_b   # static pump: sin, not J_1

ratio_list = np.arange(2, 10, 1)

tau = 2 * np.pi / w_b
n_periods = int(np.ceil(.2 / kappa / tau))   # 6: kappa_b T = 0.225
n_sub = 8
n_a = 29
n_b = 17

POINTS = [dict(label=f"eps_0 = {eps:.3f}, ed/g2 = {ed:g}", epsilon_0=float(eps),
               g2=float(g2), ratio=float(ed))
          for eps, g2 in zip(epsilon_0_arr, g2_arr) for ed in ratio_list]
COMMON = dict(n_a=n_a, n_b=n_b, n_periods=n_periods, n_sub=n_sub, kappa_b=kappa,
              precision=PRECISION)

heatmap = run_on_cluster(otoc_point, POINTS, COMMON)

fits = np.array([(np.nan, np.nan) if "error" in r else fit_log_otoc(r) for r in heatmap])
lyap_arr = (fits[:, 0] / 2).reshape(len(epsilon_0_arr), len(ratio_list))
R_sq_arr = fits[:, 1].reshape(len(epsilon_0_arr), len(ratio_list))
end_otoc_arr = np.array([np.nan if "error" in r else r["otoc"][-1] for r in heatmap]).reshape(
    len(epsilon_0_arr), len(ratio_list))
print(f"{np.isnan(lyap_arr).sum()} failed point(s) out of {lyap_arr.size}")

In [ ]:
print("g2 / 2pi (MHz):", np.round(g2_arr / 2 / np.pi * 1e3, 3))

In [ ]:
d_eps = (epsilon_0_arr[1] - epsilon_0_arr[0]) / 2
extent = (epsilon_0_arr[0] - d_eps, epsilon_0_arr[-1] + d_eps,
          ratio_list[0] - 0.5, ratio_list[-1] + 0.5)

fig, axes = plt.subplots(1, 3, figsize=(16.0, 4.4), constrained_layout=True)
for ax, arr, name in ((axes[0], lyap_arr, r"$m/2$"), (axes[1], R_sq_arr, r"$R^2$"),
                      (axes[2], end_otoc_arr, r"$C(T)$")):
    im = ax.imshow(arr.T, origin="lower", aspect="auto", extent=extent)
    ax.set_xlabel(r'$\varepsilon_p$' + ' (rad)')
    ax.set_ylabel(r'$\varepsilon_d / g_2$')
    ax.set_title(name + r', $n_a=$' + str(n_a) + ', ' + r'$n_b=$' + str(n_b))
    fig.colorbar(im, ax=ax)
plt.show()

In [ ]:
import datetime
import json
import pathlib

# Same format as the original's lyap_array_, R_sq_array_ and end_otoc_array_<date>.json:
# a list of rows, one per epsilon_0, one entry per epsilon_d / g2. Written next to
# this notebook (or the workspace root, if that is the kernel's working
# directory) under results/.
here = pathlib.Path.cwd()
out_dir = (here if here.name == "Quantum_chaos" else here / "Quantum_chaos") / "results"
out_dir.mkdir(parents=True, exist_ok=True)
date = f"{datetime.date.today():%d_%m_%Y}"

for name, arr in (("lyap_array", lyap_arr), ("R_sq_array", R_sq_arr),
                  ("end_otoc_array", end_otoc_arr)):
    out_path = out_dir / f"{name}_{date}.json"
    with open(out_path, "w") as f:
        json.dump(arr.astype(float).tolist(), f)
    print(f"wrote {out_path}")